# 24.07.24

# 리뷰분석 다음 황금 키워드 발굴
- 광고주가 원하는 키워드: 입찰가 천원 이하, 준수한 유입인원
- 입찰가 같은 경우에는 따로 확인 할 수 있는 API가 없어서 키워드 산출 후 마지막에 입찰가 확인해야할듯
- 파워링크를 통한 준수한 유입인원은 클릭률을 확인해봐야할거같다.


## 맨처음 키워드는 현재 파워링크 광고에 사용중인 메인키워들을 활용하여 API를 통해 관련 연관검색어를 모두 가져온다
## 1. 키워드 뽑고 연관검색어 및 관련 데이터 출력
### 네이버 검색 광고 API

In [7]:
import os
import sys
import urllib.request
import json
import pandas as pd
import matplotlib.pyplot as plt
import time
import random
import requests


import hashlib
import hmac
import base64


class Signature:

    @staticmethod
    def generate(timestamp, method, uri, secret_key):
        message = "{}.{}.{}".format(timestamp, method, uri)
        hash = hmac.new(bytes(secret_key, "utf-8"), bytes(message, "utf-8"), hashlib.sha256)
        
        hash.hexdigest()
        return base64.b64encode(hash.digest())
    

def get_header(method, uri, api_key, secret_key, customer_id):
    timestamp = str(round(time.time() * 1000))
    signature = Signature.generate(timestamp, method, uri, secret_key)
    
    return {'Content-Type': 'application/json; charset=UTF-8', 'X-Timestamp': timestamp, 
            'X-API-KEY': api_key, 'X-Customer': str(customer_id), 'X-Signature': signature}


def getresults(hintKeywords):

    BASE_URL = 'https://api.naver.com'
    API_KEY = '01000000001032b275c4a11023163abc8cac104361a576d200ab4a9a23d67951da6acf6b16'
    SECRET_KEY = 'AQAAAABt5vCTU15L+VltnmUaN4afh/Bvxoxg32TcV7BzFnpBdQ=='
    CUSTOMER_ID = '2961292'

    uri = '/keywordstool'
    method = 'GET'

    params={}

    params['hintKeywords']=hintKeywords
    params['showDetail']='1'

    r=requests.get(BASE_URL + uri, params=params, 
                 headers=get_header(method, uri, API_KEY, SECRET_KEY, CUSTOMER_ID))

    df = pd.DataFrame(r.json()['keywordList'])

    # 컬럼명을 변경
    df.rename(columns={
        'relKeyword': '연관키워드',
        'monthlyPcQcCnt': '월간검색수(PC)',
        'monthlyMobileQcCnt': '월간검색수(모바일)',
        'monthlyAvePcClkCnt': '월평균클릭수(PC)',
        'monthlyAveMobileClkCnt': '월평균클릭수(모바일)',
        'monthlyAvePcCtr': '월평균클릭률(PC)',
        'monthlyAveMobileCtr': '월평균클릭률(모바일)',
        'plAvgDepth': '월평균노출광고수',
        'compIdx': '경쟁정도'
    }, inplace=True)
    
    return df

### 데이터 전처리 함수
- 월간검색수 데이터가 숫자 뿐만 아니라 ">10"이런식으로 나오므로 ">"제거 후 int형 변환

In [13]:
import pandas as pd

def process_keyword(keyword):
    # getresults 함수 호출하여 데이터프레임 생성
    df = getresults(keyword)
    
    # '월간검색수(모바일)' 컬럼을 문자열로 변환
    df['월간검색수(모바일)'] = df['월간검색수(모바일)'].astype(str)
    
    # '< ' 문자를 제거
    df['월간검색수(모바일)'] = df['월간검색수(모바일)'].str.replace('< ', '')
    
    # 데이터를 숫자형(int)으로 변환
    df['월간검색수(모바일)'] = df['월간검색수(모바일)'].astype(int)
    
    # '월간검색수(모바일)' 기준으로 내림차순 정렬
    df = df.sort_values(by='월간검색수(모바일)', ascending=False)
    
    # '월간검색수(모바일)'이 10000보다 큰 값 필터링
    #result_df = df.loc[df['월간검색수(모바일)'] > 10000]

    # '경쟁정도'가 낮음 이거나 중간 필터링
    #result_df = result_df.loc[(result_df['경쟁정도']=='낮음')|(result_df['경쟁정도']=='중간')]
    
    return df


In [14]:
# 키워드 목록
key1 = ['호두정과','호두정과답례품','호두강정','호두강정답례품']


# 모든 키워드에 대한 데이터를 결합
all_data = pd.DataFrame()

for keyword in key1:
    df = process_keyword(keyword)
    
    if not df.empty:
        all_data = pd.concat([all_data, df], ignore_index=True)
    
    # Rate limiting control
    time.sleep(random.uniform(1, 3))

# 중복 제거
all_data.drop_duplicates(subset=['연관키워드'], keep='first', inplace=True)

all_data = all_data.sort_values(by='월간검색수(모바일)', ascending=False)


all_data

,연관키워드,월간검색수(PC),월간검색수(모바일),월평균클릭수(PC),월평균클릭수(모바일),월평균클릭률(PC),월평균클릭률(모바일),월평균노출광고수,경쟁정도
0,호두과자,4290,32900,35.7,344.0,0.91,1.12,10,중간
1,돌답례품,2540,30200,16.0,532.6,0.66,1.90,15,높음
2,장인약과,4590,30000,11.6,231.8,0.28,0.82,7,높음
3,호두정과,5570,28400,42.4,293.0,0.84,1.13,8,중간
4,결혼식답례품,4360,24700,21.9,306.6,0.55,1.40,15,높음
...,...,...,...,...,...,...,...,...,...
472,결혼식답례품포장,< 10,10,0.0,0.3,0.00,4.17,3,높음
473,사돈추석선물,< 10,10,0.0,0.0,0.00,0.00,0,낮음
474,명절선물견과류,< 10,10,0.0,0.0,0.00,0.00,7,높음
475,추석견과류,< 10,10,0.0,0.0,0.00,0.00,7,높음


## 2. 키워드 선별작업

In [15]:
import numpy as np

# '월간검색수(모바일)' 컬럼의 고유 값 추출
unique_mobile_search_counts = all_data['월간검색수(모바일)'].unique()

# 고유 값들을 오름차순 정렬
sorted_unique_mobile_search_counts = np.sort(unique_mobile_search_counts)

# 상위 20%에 해당하는 값의 인덱스 계산
top_20_percent_index = int(len(sorted_unique_mobile_search_counts) * 0.8)

# 상위 20%에 해당하는 값 추출
top_20_percent_value = sorted_unique_mobile_search_counts[top_20_percent_index]

# 결과 출력
print("상위 20%에 해당하는 기준 값:")
print(top_20_percent_value)


상위 20%에 해당하는 기준 값:
3530


## 검색량이 상위 20%이상 값들과 경쟁도는 낮음or중간, 월평균클릭률은 2%이상인 키워드 선정

In [16]:
test1 = all_data.loc[(all_data['월간검색수(모바일)']>= top_20_percent_value)&(all_data.경쟁정도!='높음')&(all_data['월평균클릭률(모바일)']>=2)]
test1

,연관키워드,월간검색수(PC),월간검색수(모바일),월평균클릭수(PC),월평균클릭수(모바일),월평균클릭률(PC),월평균클릭률(모바일),월평균노출광고수,경쟁정도
488,어린이집답례품,2630,19000,3.0,418.6,0.12,2.28,10,중간
491,천안호두과자,1470,11600,15.1,291.2,1.12,2.71,10,중간
26,칠순답례품,760,3790,4.7,130.4,0.66,3.67,10,중간


## 천안호두과자는 우리 상품과 관련 없으므로 제거

In [17]:
test1 = test1[test1.연관키워드 != '천안호두과자']
test1

,연관키워드,월간검색수(PC),월간검색수(모바일),월평균클릭수(PC),월평균클릭수(모바일),월평균클릭률(PC),월평균클릭률(모바일),월평균노출광고수,경쟁정도
488,어린이집답례품,2630,19000,3.0,418.6,0.12,2.28,10,중간
26,칠순답례품,760,3790,4.7,130.4,0.66,3.67,10,중간


### 네이버 검색광고 키워드 월간 예상 실적 조사 결과
- 칠순답례품 모바일 최소노출 입찰가 2650원으로 예산초과
- 어린이집답례품 모바일 최소노출 입찰가 1560원으로 예산 초과

### 원하는 키워드가 나오지 않아서 검색량 상위 30%로 기준을 늘려 다시 진행

In [18]:
# '월간검색수(모바일)' 컬럼의 고유 값 추출
unique_mobile_search_counts = all_data['월간검색수(모바일)'].unique()

# 고유 값들을 오름차순 정렬
sorted_unique_mobile_search_counts = np.sort(unique_mobile_search_counts)

# 상위 30%에 해당하는 값의 인덱스 계산
top_30_percent_index = int(len(sorted_unique_mobile_search_counts) * 0.5)

# 상위 30%에 해당하는 값 추출
top_30_percent_value = sorted_unique_mobile_search_counts[top_30_percent_index]

# 결과 출력
print("상위 30%에 해당하는 기준 값:")
print(top_30_percent_value)

상위 30%에 해당하는 기준 값:
1100


## 검색량이 상위 30%이상 값들과 경쟁도는 낮음or중간, 월평균클릭률은 2%이상인 키워드 선정
- 진행했지만 상위 20%이상 했을때와 별 차이가 없다 키워드 하나가 추가됐을뿐

In [19]:
test2 = all_data.loc[(all_data['월간검색수(모바일)']>= top_30_percent_value)&(all_data.경쟁정도!='높음')&(all_data['월평균클릭률(모바일)']>=2)]
test2

,연관키워드,월간검색수(PC),월간검색수(모바일),월평균클릭수(PC),월평균클릭수(모바일),월평균클릭률(PC),월평균클릭률(모바일),월평균노출광고수,경쟁정도
488,어린이집답례품,2630,19000,3.0,418.6,0.12,2.28,10,중간
491,천안호두과자,1470,11600,15.1,291.2,1.12,2.71,10,중간
26,칠순답례품,760,3790,4.7,130.4,0.66,3.67,10,중간
62,당뇨에좋은음식과나쁜음식,110,1420,0.4,51.8,0.35,3.96,8,중간


### 경쟁도에 대한 조건 삭제 후 다시 시도

In [20]:
test3 = all_data.loc[(all_data['월간검색수(모바일)']>= top_30_percent_value)&(all_data['월평균클릭률(모바일)']>=2)].reset_index(drop=True)
test3

,연관키워드,월간검색수(PC),월간검색수(모바일),월평균클릭수(PC),월평균클릭수(모바일),월평균클릭률(PC),월평균클릭률(모바일),월평균노출광고수,경쟁정도
0,어린이집답례품,2630,19000,3.0,418.6,0.12,2.28,10,중간
1,천안호두과자,1470,11600,15.1,291.2,1.12,2.71,10,중간
2,에너지바,2930,11400,9.1,316.2,0.34,2.98,15,높음
3,견과류종류,1170,9200,6.1,409.3,0.56,4.69,15,높음
4,구절판,1280,7760,2.6,199.5,0.22,2.76,7,높음
5,결혼답례품,1970,6490,7.0,139.6,0.39,2.26,15,높음
6,유치원답례품,500,4770,1.0,104.2,0.21,2.09,7,높음
7,견과류추천,760,4270,6.3,172.5,0.91,4.28,15,높음
8,칠순답례품,760,3790,4.7,130.4,0.66,3.67,10,중간
9,어르신간식,960,3530,5.7,84.5,0.67,2.59,7,높음


### 스토어 상품과 관련없는 키워드 제거와 위에서 실패한 키워드 제거














In [21]:
test3.drop([0,1,2,4,8,13,16,17,18,23,26,27,30,31,32,34,5,6,14,22]).reset_index(drop=True)

,연관키워드,월간검색수(PC),월간검색수(모바일),월평균클릭수(PC),월평균클릭수(모바일),월평균클릭률(PC),월평균클릭률(모바일),월평균노출광고수,경쟁정도
0,견과류종류,1170,9200,6.1,409.3,0.56,4.69,15,높음
1,견과류추천,760,4270,6.3,172.5,0.91,4.28,15,높음
2,어르신간식,960,3530,5.7,84.5,0.67,2.59,7,높음
3,장례식답례품,330,3440,3.7,117.3,1.17,3.58,15,높음
4,임산부견과류,520,3400,1.7,69.0,0.35,2.28,7,높음
5,견과류세트,760,3300,2.7,108.6,0.39,3.49,15,높음
6,승진답례품,490,2270,2.0,60.8,0.41,2.67,7,높음
7,유치원생일답례품,270,1960,3.7,55.4,1.44,2.88,7,높음
8,견과류선물세트,460,1810,2.7,36.6,0.65,2.27,15,높음
9,결혼답례선물,210,1750,3.5,87.4,1.75,5.19,15,높음


### 입찰가 확인결과 임산부견과류(700원), 견과류추천(910원), 견과류세트(860원), 견과류종류(430원) 07.24일 등록